# 10. Which TF reads a core-motif site, in which cell line (Fig. 5b,c)

Every core motif matches several candidate reader TFs (notebook 04). At the motif copies that are present in the promoter windows of all five ChIP-profiled lines (K562, HepG2, A549, GM12878, HEK293), which candidate reader actually binds, and does the strongest binder change between lines? `code/chipseq_enrichment/reader_switch/`:
1. `s1_peak_filter.py`: every TF's top-10 % peaks (by signal, within TSS ± 10 kb) from the ENCODE download (notebook 08 explains the download; HEK293 included);
2. the **motif loci**: the FIMO sites (p < 10⁻⁴) of the 43 motifs in each line's promoter windows, mapped to hg38 coordinates, and the loci shared by all five lines (271 k copies). Released on Zenodo (`fig5_reader_switch_inputs.tar.gz`, `reader_switch/out/loci/`), because rebuilding them needs the promoter-window scans of the four EpiMap lines and of HEK293 (`s2_motif_loci.py`);
3. `s3_reader_binding.py`: `bedtools intersect` of the shared loci with each reader's top peaks; the strongest binder per site and line; the per-motif switch rate between line pairs restricted to readers profiled in both;
4. `fig5bc_nature.py`: panel b (four switching motifs) and panel c (the nine switching motifs).

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/rewrite_paths.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
ENCODE = f"{ENHANCER_ROOT}/encode4_tf_chipseq"     # bin/download_encode_peaks.py --dest $ENHANCER_ROOT/encode4_tf_chipseq
def encode_status(cells):
    for c in cells:
        n = len(glob.glob(f"{ENCODE}/{c}/*/*.bed")); want = sum(1 for l in open(f"{REPO}/data/encode/encode_tf_chipseq_peaks.tsv") if l.split("\t")[0] == c)
        print(f"  {'ok ' if n == want else '-- '} {c}: {n} of {want} ENCODE peak files")
RS = f"{CODE}/chipseq_enrichment/reader_switch"; WORK = f"{ENHANCER_ROOT}/notebook_runs/fig5"; os.makedirs(WORK, exist_ok=True)
import numpy as np, pandas as pd
from IPython.display import Image, display
encode_status(["K562", "HepG2", "A549", "GM12878", "HEK293"])
print("  motif loci (Zenodo):", len(glob.glob(f"{RS}/out/loci/*.parquet")), "cell lines;", "shared loci file:", os.path.exists(f"{RS}/out/shared_loci.parquet"), "| bedtools:", shutil.which("bedtools"))
for f in ("per_motif_switch.csv", "reader_binding.parquet"): shutil.copy(f"{RS}/out/{f}", f"{WORK}/{f.replace('.', '.published.')}")

## 10.1 Top-10 % peaks of every TF (minutes)

In [ ]:
run("python s1_peak_filter.py", cwd=RS); print(pd.read_csv(f"{RS}/out/peak_filter_stats.csv").to_string(index=False))

## 10.2 The shared motif loci

In [ ]:
L = pd.read_parquet(f"{RS}/out/shared_loci.parquet"); print(f"{len(L):,} motif copies present in all five lines;", L.motif_id.nunique(), "motifs"); print(L.head(4).to_string(index=False))

## 10.3 Reader binding at the shared sites and the switch rate (minutes)

In [ ]:
run("python s3_reader_binding.py", cwd=RS)
sw = pd.read_csv(f"{RS}/out/per_motif_switch.csv"); print(sw.to_string(index=False))
old = pd.read_csv(f"{WORK}/per_motif_switch.published.csv"); print("\nidentical to the published table:", sw.round(6).equals(old.round(6)))

## 10.4 Panels b and c

In [ ]:
run("python fig5bc_nature.py", cwd=RS); display(Image(f"{RS}/out/fig5bc_nature.png", width=620))